This notebook reads the FAERS text files exactly as the FDA ships them and writes each one to a Delta table with no cleaning and no type changes yet. Every value stays a string.

What this notebook adds is tracking. Each row gets stamped with the file it came from, the moment it was ingested, and the year and quarter it belongs to. That way the origin of any record is recorded. Once the data gets here, nothing rewrites it. Every later stage reads from the data here.

In [ ]:
%pip install pyspark==3.5.0 delta-spark==3.2.0 -q

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = "/content/drive/MyDrive/faers-data-pipeline"
sys.path.insert(0, PROJECT_ROOT)

with open(os.path.join(PROJECT_ROOT, "config/pipeline_config.json")) as f:
    CONFIG = json.load(f)

from src.utils.spark_session import get_spark
spark = get_spark("FAERS-Bronze")

print(f"✓ Spark {spark.version} ready")
print(f"✓ Project root: {PROJECT_ROOT}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
Mounted at /content/drive
✓ Spark 3.5.0 ready
✓ Project root: /content/drive/MyDrive/faers-data-pipeline


This function handles any of the seven file types and any quarter. It finds the matching raw file, reads it as plain strings, adds the tracking columns, and appends the result to that table's Delta folder. 

In [ ]:
from pyspark.sql.functions import input_file_name, current_timestamp, lit
import glob

# Map file type prefixes to their glob patterns
FILE_TYPES = {
    "DEMO": "DEMO*.txt",
    "DRUG": "DRUG*.txt",
    "REAC": "REAC*.txt",
    "OUTC": "OUTC*.txt",
    "RPSR": "RPSR*.txt",
    "THER": "THER*.txt",
    "INDI": "INDI*.txt",
}

def ingest_to_bronze(spark, raw_dir, bronze_dir, file_type, year, quarter):
    """
    Read one raw FAERS file and write it to a Bronze Delta table.

    Pass the SparkSession, the raw quarter folder like data/raw/2024q1, and
    the bronze output folder. file_type is one of DEMO, DRUG, REAC, OUTC,
    RPSR, THER, or INDI. year and quarter are strings like 2024 and Q1.
    Returns the row count that got written, or 0 when no file matches.
    """
    pattern = FILE_TYPES[file_type]
    matches = glob.glob(os.path.join(raw_dir, pattern))

    if not matches:
        print(f"  ⚠ {file_type}: no file matching {pattern} in {raw_dir}")
        return 0

    filepath = matches[0]
    filename = os.path.basename(filepath)

    # Read raw. Everything comes in as strings with no schema inference.
    df = spark.read.csv(
        filepath,
        sep="$",
        header=True,
        inferSchema=False,
        encoding="ISO-8859-1",
        multiLine=True,
        mode="PERMISSIVE",
    )

    # Add metadata columns for lineage tracking
    df_bronze = (df
        .withColumn("_source_file", lit(filename))
        .withColumn("_ingestion_ts", current_timestamp())
        .withColumn("_year", lit(year))
        .withColumn("_quarter", lit(quarter))
    )

    # Write to Delta in append mode so multiple quarters accumulate.
    output_path = os.path.join(bronze_dir, file_type.lower())
    df_bronze.write.format("delta") \
        .mode("append") \
        .partitionBy("_year", "_quarter") \
        .save(output_path)

    row_count = df_bronze.count()
    print(f"  ✓ {file_type}: {row_count:,} rows from {filename}")
    return row_count

print("✓ ingest_to_bronze() function defined")

✓ ingest_to_bronze() function defined


This runs the function across all seven file types for 2024 Q1. The loop keeps a running total as well.

In [ ]:
YEAR = "2024"
QUARTER = "Q1"
raw_dir = os.path.join(PROJECT_ROOT, f"data/raw/{YEAR.lower()}q1")
bronze_dir = os.path.join(PROJECT_ROOT, "data/bronze")

print(f"Ingesting {YEAR} {QUARTER} from: {raw_dir}")
print(f"Writing Bronze to: {bronze_dir}\n")

total_rows = 0
results = {}

for file_type in FILE_TYPES:
    count = ingest_to_bronze(spark, raw_dir, bronze_dir, file_type, YEAR, QUARTER)
    results[file_type] = count
    total_rows += count

print(f"\n{'='*40}")
print(f"Bronze ingestion complete!")
print(f"Total rows across all tables: {total_rows:,}")
print(f"{'='*40}")

Ingesting 2024 Q1 from: /content/drive/MyDrive/faers-data-pipeline/data/raw/2024q1
Writing Bronze to: /content/drive/MyDrive/faers-data-pipeline/data/bronze

  ✓ DEMO: 406,184 rows from DEMO24Q1.txt
  ✓ DRUG: 1,909,327 rows from DRUG24Q1.txt
  ✓ REAC: 1,445,416 rows from REAC24Q1.txt
  ✓ OUTC: 295,044 rows from OUTC24Q1.txt
  ✓ RPSR: 12,381 rows from RPSR24Q1.txt
  ✓ THER: 594,449 rows from THER24Q1.txt
  ✓ INDI: 1,186,115 rows from INDI24Q1.txt

Bronze ingestion complete!
Total rows across all tables: 5,848,916


This reads each Bronze table back and reports its row count, its column count, and how many partitions it holds.

In [ ]:
bronze_dir = os.path.join(PROJECT_ROOT, "data/bronze")

print(f"{'Table':<8} {'Rows':>10} {'Columns':>9} {'Partitions':>12}")
print("-" * 43)

for file_type in FILE_TYPES:
    delta_path = os.path.join(bronze_dir, file_type.lower())

    if not os.path.exists(delta_path):
        print(f"{file_type:<8} {'NOT FOUND':>10}")
        continue

    df = spark.read.format("delta").load(delta_path)
    row_count = df.count()
    # Subtract 4 metadata columns from total
    data_cols = len(df.columns) - 4
    partitions = df.select("_year", "_quarter").distinct().count()
    print(f"{file_type:<8} {row_count:>10,} {data_cols:>9} {partitions:>12}")

print()

# Show metadata columns on DEMO as proof of lineage tracking
print("DEMO metadata columns (first 5 rows):")
demo_bronze = spark.read.format("delta").load(
    os.path.join(bronze_dir, "demo"))
demo_bronze.select(
    "primaryid", "_source_file", "_ingestion_ts", "_year", "_quarter"
).limit(5).toPandas()

Table          Rows   Columns   Partitions
-------------------------------------------
DEMO        406,184        25            1
DRUG      1,909,327        20            1
REAC      1,445,416         4            1
OUTC        295,044         3            1
RPSR         12,381         3            1
THER        594,449         7            1
INDI      1,186,115         4            1

DEMO metadata columns (first 5 rows):


,primaryid,_source_file,_ingestion_ts,_year,_quarter
0,1001678125,DEMO24Q1.txt,2026-03-25 00:52:06.409176,2024,Q1
1,1002872124,DEMO24Q1.txt,2026-03-25 00:52:06.409176,2024,Q1
2,100293663,DEMO24Q1.txt,2026-03-25 00:52:06.409176,2024,Q1
3,1005450710,DEMO24Q1.txt,2026-03-25 00:52:06.409176,2024,Q1
4,1005762118,DEMO24Q1.txt,2026-03-25 00:52:06.409176,2024,Q1


This checks the main tables and finds any data column that is not a string.  

In [ ]:
from pyspark.sql.types import StringType

bronze_dir = os.path.join(PROJECT_ROOT, "data/bronze")

for file_type in ["DEMO", "DRUG", "REAC"]:
    delta_path = os.path.join(bronze_dir, file_type.lower())
    df = spark.read.format("delta").load(delta_path)

    # Check data columns (skip metadata columns starting with _)
    data_fields = [f for f in df.schema.fields if not f.name.startswith("_")]
    non_string = [f for f in data_fields if f.dataType != StringType()]

    if non_string:
        print(f"⚠ {file_type}: these columns are NOT string:")
        for f in non_string:
            print(f"    {f.name}: {f.dataType}")
    else:
        print(f"✓ {file_type}: all {len(data_fields)} data columns are StringType")

✓ DEMO: all 25 data columns are StringType
✓ DRUG: all 20 data columns are StringType
✓ REAC: all 4 data columns are StringType


Delta keeps a full history of every write. This shows what tables changed and when and there are also version numbers as well.  For example this shows the history for DEMO and reads version 0.

In [ ]:
from delta.tables import DeltaTable

bronze_dir = os.path.join(PROJECT_ROOT, "data/bronze")
demo_path = os.path.join(bronze_dir, "demo")

delta_table = DeltaTable.forPath(spark, demo_path)

print("DEMO Bronze — Delta history:")
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics.numOutputRows",
    "operationMetrics.numFiles"
).show(truncate=False)

# Read version 0 (the initial load)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(demo_path)
print(f"Version 0 row count: {df_v0.count():,}")

DEMO Bronze — Delta history:
+-------+-------------------+---------+-------------+--------+
|version|timestamp          |operation|numOutputRows|numFiles|
+-------+-------------------+---------+-------------+--------+
|0      |2026-03-25 00:52:28|WRITE    |406184       |1       |
+-------+-------------------+---------+-------------+--------+

Version 0 row count: 406,184


This wrapper loads a whole quarter in one call. When downloading later quarters, this function will be used for each one.

In [ ]:
def ingest_quarter_to_bronze(spark, project_root, year, quarter_num):
    """
    Ingest all seven FAERS files for one quarter into Bronze Delta tables.

    Pass the SparkSession, the project root, the year as a string like 2024,
    and just the quarter number like 1. It loops over every file type and
    returns a dict that maps each file type to the number of rows it wrote.
    """
    quarter_label = f"Q{quarter_num}"
    raw_dir = os.path.join(project_root, f"data/raw/{year}q{quarter_num}")
    bronze_dir = os.path.join(project_root, "data/bronze")

    if not os.path.exists(raw_dir):
        print(f"⚠ Raw directory not found: {raw_dir}")
        return {}

    print(f"{'='*50}")
    print(f"Ingesting {year} {quarter_label}")
    print(f"{'='*50}")

    results = {}
    for file_type in FILE_TYPES:
        count = ingest_to_bronze(spark, raw_dir, bronze_dir,
                                  file_type, year, quarter_label)
        results[file_type] = count

    total = sum(results.values())
    print(f"\n  Total: {total:,} rows for {year} {quarter_label}")
    return results

print("✓ ingest_quarter_to_bronze() function defined")
print("\nTo ingest future quarters:")
print('  ingest_quarter_to_bronze(spark, PROJECT_ROOT, "2024", "2")')

✓ ingest_quarter_to_bronze() function defined

To ingest future quarters:
  ingest_quarter_to_bronze(spark, PROJECT_ROOT, "2024", "2")


Now the pipeline is also stored in bronze_ingestion.py